In [13]:
# ============================================================
# BitFit 复现实验：RTE + bert-base-cased
# 方法：Full FT / Classifier-only / BitFit
#       Partial BitFit (query+intermediate) / Partial BitFit (query-only)
# 种子：42, 123, 2026
# 额外：BitFit seed=42 学习率扫描 1e-4 / 5e-4 / 1e-3
# ============================================================

import os
import gc
import json
import time
import random
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.utils.data import DataLoader

# ============================================================
# 1. 配置
# ============================================================

MODEL_NAME = "bert-base-cased"
TASK_NAME = "rte"

MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 4
SEEDS = [42, 123, 2026]

LR_SWEEP_LRS = [1e-4, 5e-4, 1e-3]
LR_SWEEP_SEED = 42

PARTIAL_BITFIT_LR = 5e-3

ROOT = Path("/content/drive/MyDrive/nlp实践/BitFit_results")
MODEL_DIR = ROOT / "models"
RESULT_DIR = ROOT / "results"
PRED_DIR = RESULT_DIR / "predictions"

for p in [MODEL_DIR, RESULT_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RESULT_JSON = RESULT_DIR / "training_results.json"
SUMMARY_CSV = RESULT_DIR / "summary.csv"
PARAM_AUDIT_CSV = RESULT_DIR / "parameter_audit.csv"
GRAD_AUDIT_CSV = RESULT_DIR / "gradient_audit.csv"
THREE_SEED_STAT_CSV = RESULT_DIR / "three_seed_statistics.csv"
LR_SWEEP_SUMMARY_CSV = RESULT_DIR / "bitfit_lr_sweep_summary.csv"
LR_ERROR_COMPARISON_CSV = RESULT_DIR / "bitfit_lr_sweep_error_comparison.csv"
LR_ERROR_CHANGES_CSV = RESULT_DIR / "bitfit_lr_sweep_error_changes.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

METHODS = {
    "Full FT": {"lr": 2e-5},
    "Classifier-only": {"lr": 1e-3},
    "BitFit": {"lr": 5e-4},
    "Partial BitFit (query+intermediate)": {"lr": PARTIAL_BITFIT_LR},
    "Partial BitFit (query-only)": {"lr": PARTIAL_BITFIT_LR},
}

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Device:", DEVICE)

if DEVICE.type != "cuda":
    raise RuntimeError("没有检测到 GPU，请在 Colab 中选择 GPU 运行时。")

print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. 随机种子
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ============================================================
# 3. 数据
# ============================================================

print("\n加载 GLUE RTE 数据集...")
raw_dataset = load_dataset("nyu-mll/glue", "rte")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["sentence1"],
        examples["sentence2"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

tokenized_dataset = raw_dataset.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing RTE"
)

model_columns = ["input_ids", "attention_mask", "token_type_ids", "label"]

for split in ["train", "validation"]:
    tokenized_dataset[split].set_format(type="torch", columns=model_columns)

tokenized_dataset["test"].set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids"]
)

train_dataset = tokenized_dataset["train"]
val_dataset = tokenized_dataset["validation"]
test_dataset = tokenized_dataset["test"]

assert len(train_dataset) == 2490
assert len(val_dataset) == 277

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ============================================================
# 4. 可训练参数规则
# ============================================================

def configure_trainable_parameters(model, method):
    for name, param in model.named_parameters():
        if method == "Full FT":
            trainable = True

        elif method == "Classifier-only":
            trainable = name.startswith("classifier.")

        elif method == "BitFit":
            trainable = (
                (name.endswith(".bias") and not name.startswith("classifier."))
                or name.startswith("classifier.")
            )

        elif method == "Partial BitFit (query+intermediate)":
            trainable = (
                name.startswith("classifier.")
                or name.endswith("attention.self.query.bias")
                or name.endswith("intermediate.dense.bias")
                or name.endswith("pooler.dense.bias")
            )

        elif method == "Partial BitFit (query-only)":
            trainable = (
                name.startswith("classifier.")
                or name.endswith("attention.self.query.bias")
            )

        else:
            raise ValueError(method)

        param.requires_grad = trainable

    trainable_items = [
        (name, param)
        for name, param in model.named_parameters()
        if param.requires_grad
    ]

    assert len(trainable_items) > 0

    if method != "Full FT":
        classifier_names = {
            n for n, _ in trainable_items if n.startswith("classifier.")
        }
        assert "classifier.weight" in classifier_names
        assert "classifier.bias" in classifier_names

    return trainable_items

# ============================================================
# 5. 参数与梯度审计
# ============================================================

def snapshot_parameters(model, trainable_only=False):
    snapshot = {}
    for name, param in model.named_parameters():
        if trainable_only and not param.requires_grad:
            continue
        snapshot[name] = param.detach().cpu().clone()
    return snapshot

def compare_parameters(model, before):
    updated, unchanged = [], []
    for name, param in model.named_parameters():
        if name not in before:
            continue
        old = before[name]
        new = param.detach().cpu()
        if torch.equal(old, new):
            unchanged.append(name)
        else:
            updated.append(name)
    return updated, unchanged

def check_gradient_status(model):
    with_grad, missing_grad, zero_grad, nonfinite_grad = [], [], [], []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if param.grad is None:
            missing_grad.append(name)
            continue

        grad = param.grad.detach()
        if not torch.isfinite(grad).all():
            nonfinite_grad.append(name)
        if torch.count_nonzero(grad).item() == 0:
            zero_grad.append(name)
        with_grad.append(name)

    return {
        "gradient_tensors": len(with_grad),
        "missing_gradient_tensors": len(missing_grad),
        "zero_gradient_tensors": len(zero_grad),
        "nonfinite_gradient_tensors": len(nonfinite_grad),
    }

def assert_updated_rule(method, updated_names):
    if method == "Classifier-only":
        bad = [n for n in updated_names if not n.startswith("classifier.")]

    elif method == "BitFit":
        bad = [
            n for n in updated_names
            if not (n.startswith("classifier.") or n.endswith(".bias"))
        ]

    elif method == "Partial BitFit (query+intermediate)":
        bad = [
            n for n in updated_names
            if not (
                n.startswith("classifier.")
                or n.endswith("attention.self.query.bias")
                or n.endswith("intermediate.dense.bias")
                or n.endswith("pooler.dense.bias")
            )
        ]

    elif method == "Partial BitFit (query-only)":
        bad = [
            n for n in updated_names
            if not (
                n.startswith("classifier.")
                or n.endswith("attention.self.query.bias")
            )
        ]

    else:
        bad = []

    if bad:
        raise RuntimeError(
            f"{method} 实际更新参数不符合规则，违规参数示例：{bad[:10]}"
        )

# ============================================================
# 6. 评估与预测
# ============================================================

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, correct = 0, 0
    losses = []
    all_predictions, all_labels = [], []

    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop("label")
        outputs = model(**batch)
        loss = F.cross_entropy(outputs.logits, labels)
        predictions = outputs.logits.argmax(dim=-1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()
        losses.append(loss.item())
        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return {
        "accuracy": correct / total,
        "loss": float(np.mean(losses)),
        "predictions": all_predictions,
        "labels": all_labels,
    }

@torch.no_grad()
def predict_test(model, loader):
    model.eval()
    predictions = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        preds = outputs.logits.argmax(dim=-1)
        predictions.extend(preds.cpu().tolist())
    return predictions

# ============================================================
# 7. 结果保存
# ============================================================

def make_run_id(method, seed, lr):
    return f"{method}_seed{int(seed)}_lr{lr}"

def load_results():
    if RESULT_JSON.exists():
        with open(RESULT_JSON, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

def ensure_run_id(records):
    for r in records:
        if "run_id" not in r:
            r["run_id"] = make_run_id(r["method"], r["seed"], r["lr"])
        if "experiment_group" not in r:
            r["experiment_group"] = "main"
    return records

all_results = ensure_run_id(load_results())

def save_results():
    tmp = RESULT_JSON.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)
    os.replace(tmp, RESULT_JSON)

def experiment_done(run_id):
    return any(
        r.get("run_id") == run_id and r.get("status") == "completed"
        for r in all_results
    )

# ============================================================
# 8. 单次训练
# ============================================================

def train_one(method, seed, lr_override=None, experiment_group="main"):
    lr = METHODS[method]["lr"] if lr_override is None else lr_override
    run_id = make_run_id(method, seed, lr)

    print("\n" + "=" * 90)
    print(f"METHOD: {method} | SEED: {seed} | LR: {lr}")
    print(f"RUN_ID: {run_id} | GROUP: {experiment_group}")
    print("=" * 90)

    set_seed(seed)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    ).to(DEVICE)

    trainable_items = configure_trainable_parameters(model, method)
    trainable_names = [name for name, _ in trainable_items]

    trainable_params = sum(p.numel() for _, p in trainable_items)
    total_params = sum(p.numel() for p in model.parameters())

    print("Total parameters:", total_params)
    print("Trainable parameters:", trainable_params)
    print("Trainable ratio:", trainable_params / total_params)
    print("Trainable tensors:", len(trainable_items))

    before_all = snapshot_parameters(model)

    optimizer = torch.optim.AdamW(
        [p for _, p in trainable_items],
        lr=lr,
        weight_decay=0.01
    )

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    start_time = time.time()
    history = []
    first_batch_gradient_audit = None

    best_val_acc = -1.0
    best_epoch = -1
    best_state = None

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        seen = 0

        for step, batch in enumerate(train_loader):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = batch.pop("label")

            optimizer.zero_grad(set_to_none=True)
            outputs = model(**batch)
            loss = F.cross_entropy(outputs.logits, labels)

            if not torch.isfinite(loss):
                raise RuntimeError("Training loss 出现 NaN/Inf")

            loss.backward()

            if epoch == 0 and step == 0:
                first_batch_gradient_audit = check_gradient_status(model)

            torch.nn.utils.clip_grad_norm_(
                [p for _, p in trainable_items],
                max_norm=1.0
            )

            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            seen += 1

        train_loss = running_loss / max(seen, 1)
        val_result = evaluate(model, val_loader)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "validation_loss": val_result["loss"],
            "validation_accuracy": val_result["accuracy"],
        })

        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Train loss={train_loss:.4f} | "
            f"Val loss={val_result['loss']:.4f} | "
            f"Val acc={val_result['accuracy']:.4f}"
        )

        if val_result["accuracy"] > best_val_acc:
            best_val_acc = val_result["accuracy"]
            best_epoch = epoch + 1
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

    elapsed = time.time() - start_time
    peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)

    if best_state is None:
        raise RuntimeError("没有得到有效的最佳模型")

    model.load_state_dict(best_state)
    del best_state
    gc.collect()

    updated, unchanged = compare_parameters(model, before_all)
    updated_trainable = [n for n in updated if n in trainable_names]
    unchanged_trainable = [n for n in unchanged if n in trainable_names]

    frozen_names = [
        n for n, p in model.named_parameters() if not p.requires_grad
    ]
    frozen_changed = [n for n in updated if n in frozen_names]

    if frozen_changed:
        raise RuntimeError("发现冻结参数发生变化：" + str(frozen_changed[:10]))

    if not updated_trainable:
        raise RuntimeError("没有任何可训练参数发生变化")

    assert_updated_rule(method, updated_trainable)

    folder = method.replace(" ", "_") + f"_seed{seed}_lr{lr}"
    model_path = MODEL_DIR / folder
    model_path.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)

    del model
    gc.collect()
    torch.cuda.empty_cache()

    reloaded_model = AutoModelForSequenceClassification.from_pretrained(
        model_path
    ).to(DEVICE)
    reloaded_model.eval()

    reloaded_val = evaluate(reloaded_model, val_loader)

    if abs(reloaded_val["accuracy"] - best_val_acc) > 1e-6:
        raise RuntimeError("重新加载模型后的验证准确率与训练时不一致")

    val_pred_df = pd.DataFrame({
        "sample_id": range(len(reloaded_val["labels"])),
        "true_label_id": reloaded_val["labels"],
        "predicted_label_id": reloaded_val["predictions"],
        "correct": [
            int(a == b)
            for a, b in zip(reloaded_val["labels"], reloaded_val["predictions"])
        ],
    })
    val_pred_df["method"] = method
    val_pred_df["seed"] = seed
    val_pred_df["lr"] = lr
    val_pred_df["run_id"] = run_id

    val_pred_path = PRED_DIR / f"{folder}_validation.csv"
    val_pred_df.to_csv(val_pred_path, index=False, encoding="utf-8-sig")

    test_predictions = predict_test(reloaded_model, test_loader)
    test_pred_df = pd.DataFrame({
        "sample_id": range(len(test_predictions)),
        "predicted_label_id": test_predictions,
        "predicted_label": [
            "entailment" if x == 0 else "not_entailment"
            for x in test_predictions
        ],
        "method": method,
        "seed": seed,
        "lr": lr,
        "run_id": run_id,
    })
    test_pred_path = PRED_DIR / f"{folder}_test.csv"
    test_pred_df.to_csv(test_pred_path, index=False, encoding="utf-8-sig")

    record = {
        "run_id": run_id,
        "experiment_group": experiment_group,
        "method": method,
        "seed": seed,
        "lr": lr,
        "task": TASK_NAME,
        "model": MODEL_NAME,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "max_length": MAX_LENGTH,

        "total_params": total_params,
        "trainable_params": trainable_params,
        "trainable_ratio": trainable_params / total_params,
        "trainable_tensors": len(trainable_names),

        "best_epoch": best_epoch,
        "best_val_acc": best_val_acc,
        "final_val_acc": history[-1]["validation_accuracy"],

        "elapsed_seconds": elapsed,
        "peak_memory_gib": peak_memory,

        "updated_trainable_tensors": len(updated_trainable),
        "unchanged_trainable_tensors": len(unchanged_trainable),
        "frozen_tensors": len(frozen_names),
        "frozen_changed_tensors": len(frozen_changed),

        "gradient_tensors": first_batch_gradient_audit["gradient_tensors"],
        "missing_gradient_tensors": first_batch_gradient_audit["missing_gradient_tensors"],
        "zero_gradient_tensors": first_batch_gradient_audit["zero_gradient_tensors"],
        "nonfinite_gradient_tensors": first_batch_gradient_audit["nonfinite_gradient_tensors"],
        "gradient_check_pass": (
            first_batch_gradient_audit["missing_gradient_tensors"] == 0
            and first_batch_gradient_audit["nonfinite_gradient_tensors"] == 0
        ),

        "history": history,
        "trainable_names": trainable_names,
        "updated_names": updated_trainable,
        "model_path": str(model_path),
        "validation_prediction_path": str(val_pred_path),
        "test_prediction_path": str(test_pred_path),
        "status": "completed",
    }

    del reloaded_model
    gc.collect()
    torch.cuda.empty_cache()

    return record

# ============================================================
# 9. 汇总表构建
# ============================================================

def to_json_str(x):
    return x if isinstance(x, str) else json.dumps(x, ensure_ascii=False)

def build_param_df(records):
    rows = []
    for r in records:
        rows.append({
            "run_id": r["run_id"],
            "experiment_group": r.get("experiment_group", "main"),
            "method": r["method"],
            "seed": r["seed"],
            "lr": r["lr"],
            "total_params": r["total_params"],
            "trainable_params": r["trainable_params"],
            "trainable_ratio": r["trainable_params"] / r["total_params"],
            "trainable_tensors": r["trainable_tensors"],
            "updated_trainable_tensors": r["updated_trainable_tensors"],
            "unchanged_trainable_tensors": r["unchanged_trainable_tensors"],
            "frozen_tensors": r["frozen_tensors"],
            "frozen_changed_tensors": r["frozen_changed_tensors"],
            "trainable_names": to_json_str(r["trainable_names"]),
            "updated_names": to_json_str(r["updated_names"]),
        })
    return pd.DataFrame(rows)

def build_grad_df(records):
    rows = []
    for r in records:
        rows.append({
            "run_id": r["run_id"],
            "experiment_group": r.get("experiment_group", "main"),
            "method": r["method"],
            "seed": r["seed"],
            "lr": r["lr"],
            "gradient_tensors": r["gradient_tensors"],
            "missing_gradient_tensors": r["missing_gradient_tensors"],
            "zero_gradient_tensors": r["zero_gradient_tensors"],
            "nonfinite_gradient_tensors": r["nonfinite_gradient_tensors"],
            "gradient_check_pass": r["gradient_check_pass"],
        })
    return pd.DataFrame(rows)

def build_lr_error_comparison(lr_records):
    merged = None
    for r in sorted(lr_records, key=lambda x: float(x["lr"])):
        lr = r["lr"]
        df = pd.read_csv(r["validation_prediction_path"])
        d = df[["sample_id", "true_label_id", "predicted_label_id", "correct"]].copy()
        suffix = f"lr{lr}"
        d = d.rename(columns={
            "predicted_label_id": f"pred_{suffix}",
            "correct": f"correct_{suffix}",
        })
        if merged is None:
            merged = d
        else:
            merged = merged.merge(
                d, on=["sample_id", "true_label_id"], how="outer"
            )

    correct_cols = [c for c in merged.columns if c.startswith("correct_lr")]
    merged["any_error"] = (merged[correct_cols] == 0).any(axis=1)
    merged["all_error"] = (merged[correct_cols] == 0).all(axis=1)
    merged.to_csv(LR_ERROR_COMPARISON_CSV, index=False, encoding="utf-8-sig")

    base_lr = 5e-4
    base_col = f"correct_lr{base_lr}"

    rows = []
    for lr in LR_SWEEP_LRS:
        col = f"correct_lr{lr}"
        wrong = int((merged[col] == 0).sum())
        fixed = int(((merged[base_col] == 0) & (merged[col] == 1)).sum())
        broken = int(((merged[base_col] == 1) & (merged[col] == 0)).sum())
        rows.append({
            "lr": lr,
            "wrong_samples": wrong,
            "correct_samples": int((merged[col] == 1).sum()),
            "fixed_vs_5e-4": fixed,
            "broken_vs_5e-4": broken,
            "net_change_vs_5e-4": fixed - broken,
        })

    changes_df = pd.DataFrame(rows)
    changes_df.to_csv(LR_ERROR_CHANGES_CSV, index=False, encoding="utf-8-sig")
    return merged, changes_df

# ============================================================
# 10. 实验计划
# ============================================================

MAIN_PLAN = [
    (method, seed, METHODS[method]["lr"], "main")
    for method in METHODS
    for seed in SEEDS
]

LR_SWEEP_PLAN = [
    ("BitFit", LR_SWEEP_SEED, lr, "lr_sweep")
    for lr in LR_SWEEP_LRS
]

seen = set()
UNIQUE_PLAN = []
for method, seed, lr, group in MAIN_PLAN + LR_SWEEP_PLAN:
    rid = make_run_id(method, seed, lr)
    if rid in seen:
        continue
    seen.add(rid)
    UNIQUE_PLAN.append((method, seed, lr, group))

MAIN_IDS = {
    make_run_id(method, seed, METHODS[method]["lr"])
    for method in METHODS
    for seed in SEEDS
}
LR_SWEEP_IDS = {
    make_run_id("BitFit", LR_SWEEP_SEED, lr)
    for lr in LR_SWEEP_LRS
}

print("\n实验计划总数（去重后）：", len(UNIQUE_PLAN))
for item in UNIQUE_PLAN:
    print("  ", item)

# ============================================================
# 11. 执行实验
# ============================================================

for method, seed, lr, group in UNIQUE_PLAN:
    rid = make_run_id(method, seed, lr)

    if experiment_done(rid):
        print(f"\n跳过已完成：{rid}")
        continue

    record = train_one(
        method,
        seed,
        lr_override=lr,
        experiment_group=group
    )
    all_results.append(record)
    save_results()
    print("\n实验成功保存：", rid)

# ============================================================
# 12. 汇总输出
# ============================================================

completed = [r for r in all_results if r.get("status") == "completed"]
summary_df = pd.DataFrame(completed)
summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")

parameter_df = build_param_df(completed)
parameter_df.to_csv(PARAM_AUDIT_CSV, index=False, encoding="utf-8-sig")

gradient_df = build_grad_df(completed)
gradient_df.to_csv(GRAD_AUDIT_CSV, index=False, encoding="utf-8-sig")

main_records = [r for r in completed if r["run_id"] in MAIN_IDS]
summary_main_df = pd.DataFrame(main_records)

statistics_df = (
    summary_main_df
    .groupby("method")
    .agg(
        mean_best_val_acc=("best_val_acc", "mean"),
        std_best_val_acc=("best_val_acc", "std"),
        mean_final_val_acc=("final_val_acc", "mean"),
        std_final_val_acc=("final_val_acc", "std"),
        mean_trainable_params=("trainable_params", "mean"),
        mean_elapsed_seconds=("elapsed_seconds", "mean"),
        mean_peak_memory_gib=("peak_memory_gib", "mean"),
    )
    .reset_index()
)
statistics_df.to_csv(THREE_SEED_STAT_CSV, index=False, encoding="utf-8-sig")

lr_records = sorted(
    [r for r in completed if r["run_id"] in LR_SWEEP_IDS],
    key=lambda x: float(x["lr"])
)

lr_summary_df = pd.DataFrame([
    {
        "lr": r["lr"],
        "best_val_acc": r["best_val_acc"],
        "final_val_acc": r["final_val_acc"],
        "trainable_params": r["trainable_params"],
        "trainable_tensors": r["trainable_tensors"],
        "updated_trainable_tensors": r["updated_trainable_tensors"],
        "elapsed_seconds": r["elapsed_seconds"],
        "peak_memory_gib": r["peak_memory_gib"],
        "run_id": r["run_id"],
    }
    for r in lr_records
])
lr_summary_df.to_csv(LR_SWEEP_SUMMARY_CSV, index=False, encoding="utf-8-sig")

lr_error_merged, lr_error_changes_df = build_lr_error_comparison(lr_records)

# ============================================================
# 13. 最终检查
# ============================================================

print("\n" + "=" * 90)
print("全部实验完成")
print("=" * 90)

print("\n完成实验数量：", len(completed))
print("计划实验数量：", len(UNIQUE_PLAN))
print("主实验数量：", len(main_records), "/ 15")
print("学习率扫描数量：", len(lr_records), f"/ {len(LR_SWEEP_LRS)}")

assert len(completed) == len(UNIQUE_PLAN), "完成实验数量与计划不一致"
assert len(main_records) == 15, "主实验不是 15 组"
assert len(lr_records) == len(LR_SWEEP_LRS), "学习率扫描不完整"

assert (parameter_df["frozen_changed_tensors"] == 0).all(), "存在冻结参数变化"
assert (parameter_df["updated_trainable_tensors"] > 0).all(), "存在可训练参数完全未更新"
assert (gradient_df["missing_gradient_tensors"] == 0).all(), "存在缺失梯度"
assert (gradient_df["nonfinite_gradient_tensors"] == 0).all(), "存在非有限梯度"

for r in completed:
    assert_updated_rule(r["method"], r["updated_names"])

print("\n主实验三种子统计：")
print(statistics_df.to_string(index=False))

print("\nBitFit 学习率扫描：")
print(lr_summary_df.to_string(index=False))

print("\nBitFit 学习率错误样本变化（相对 lr=5e-4）：")
print(lr_error_changes_df.to_string(index=False))

print("\n参数审计（Partial BitFit (query+intermediate)）：")
print(
    parameter_df[
        parameter_df["method"] == "Partial BitFit (query+intermediate)"
    ][
        ["run_id", "trainable_params", "trainable_tensors",
         "updated_trainable_tensors", "frozen_changed_tensors"]
    ].to_string(index=False)
)

print("\n全部审计断言通过。")
print("\n结果目录：", RESULT_DIR)
print("模型目录：", MODEL_DIR)
print("预测目录：", PRED_DIR)
print("\n重点文件：")
print("  ", SUMMARY_CSV)
print("  ", THREE_SEED_STAT_CSV)
print("  ", PARAM_AUDIT_CSV)
print("  ", GRAD_AUDIT_CSV)
print("  ", LR_SWEEP_SUMMARY_CSV)
print("  ", LR_ERROR_COMPARISON_CSV)
print("  ", LR_ERROR_CHANGES_CSV)

Python: 3.13.16
PyTorch: 2.14.1+cu130
CUDA: 13.0
Device: cuda
GPU: Tesla T4

加载 GLUE RTE 数据集...
Train: 2490
Validation: 277
Test: 3000

实验计划总数（去重后）： 17
   ('Full FT', 42, 2e-05, 'main')
   ('Full FT', 123, 2e-05, 'main')
   ('Full FT', 2026, 2e-05, 'main')
   ('Classifier-only', 42, 0.001, 'main')
   ('Classifier-only', 123, 0.001, 'main')
   ('Classifier-only', 2026, 0.001, 'main')
   ('BitFit', 42, 0.0005, 'main')
   ('BitFit', 123, 0.0005, 'main')
   ('BitFit', 2026, 0.0005, 'main')
   ('Partial BitFit (query+intermediate)', 42, 0.005, 'main')
   ('Partial BitFit (query+intermediate)', 123, 0.005, 'main')
   ('Partial BitFit (query+intermediate)', 2026, 0.005, 'main')
   ('Partial BitFit (query-only)', 42, 0.005, 'main')
   ('Partial BitFit (query-only)', 123, 0.005, 'main')
   ('Partial BitFit (query-only)', 2026, 0.005, 'main')
   ('BitFit', 42, 0.0001, 'lr_sweep')
   ('BitFit', 42, 0.001, 'lr_sweep')

跳过已完成：Full FT_seed42_lr2e-05

跳过已完成：Full FT_seed123_lr2e-05

跳过已完成：Full FT_seed